# 1 - Install dependencies (networkx, pyvis, louvain, sklearn)

In [1]:
# Title: Install exact versions that match Colab's stack (and fix IPython)
!pip install --quiet --force-reinstall \
  ipython==7.34.0 \
  numpy==1.26.4 \
  pandas==2.2.2 \
  networkx==3.3 \
  pyvis==0.3.2 \
  python-louvain==0.16 \
  scikit-learn==1.5.1 \
  tqdm==4.67.1

print("✅ Installed compatible versions. Now go to: Runtime > Restart runtime")


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.2.1 which is incompatible.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
umap-learn 0.5.9.post2 requires scikit-learn>=1.6, but you have scikit-learn 1.5.1 which is incompatible.
✅ Installed compatible versions. Now go to: Runtime > Restart runtime


In [2]:
# Title: Verify imports after restart
import numpy as np, pandas as pd, networkx as nx, sklearn
import community  # python-louvain
from pyvis.network import Network

print("✅ Imports OK")
print("IPython compatible")
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("networkx:", nx.__version__)
print("sklearn:", sklearn.__version__)


✅ Imports OK
IPython compatible
numpy: 1.26.4
pandas: 2.2.2
networkx: 3.3
sklearn: 1.5.1


#2 - Mount Drive & set paths / knobs

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
!ls

bin			    kaggle		      opt		 sys
boot			    lib			      proc		 tmp
content			    lib32		      python-apt	 tools
cuda-keyring_1.1-1_all.deb  lib64		      python-apt.tar.xz  usr
datalab			    libx32		      root		 var
dev			    media		      run
etc			    mnt			      sbin
home			    NGC-DL-CONTAINER-LICENSE  srv


In [23]:
cd /content/drive/MyDrive/fraud2

/content/drive/MyDrive/fraud2


In [31]:
!ls fraud2/PS_20174392719_1491204439457_log.csv

fraud2/PS_20174392719_1491204439457_log.csv


In [32]:
!ls fraud2/PS_20174392719_1491204439457_log.csv
CSV_PATH   = "fraud2/PS_20174392719_1491204439457_log.csv"  # ← change if needed
# Use either:
!ls $CSV_PATH
N_ROWS     = None    # None = full; or set e.g., 500000 to sample for quick runs
PRINT_HEAD = 5

# Graph/time window config (PaySim step unit = hours)
WINDOW_MIN_H = 0
WINDOW_MAX_H = 200   # keep small for fast demo; increase later

# Scoring sample size (how many txns to score for demo)
SCORE_LIMIT = 20000  # increase after demo

print("📁 CSV_PATH:", CSV_PATH)
print(f"⏱️ Window: [{WINDOW_MIN_H}, {WINDOW_MAX_H}] hours | SCORE_LIMIT={SCORE_LIMIT}")


fraud2/PS_20174392719_1491204439457_log.csv
fraud2/PS_20174392719_1491204439457_log.csv
📁 CSV_PATH: fraud2/PS_20174392719_1491204439457_log.csv
⏱️ Window: [0, 200] hours | SCORE_LIMIT=20000


#3 - Load PaySim (typed) & preview

In [33]:
!ls $CSV_PATH

fraud2/PS_20174392719_1491204439457_log.csv


In [34]:
import hashlib

def hash_id(x):  # short privacy-safe IDs for graph nodes
    return hashlib.md5(str(x).encode()).hexdigest()[:8]

dtypes = {
    "type": "category",
    "nameOrig": "string",
    "nameDest": "string",
    "amount": "float32",
    "oldbalanceOrg": "float32",
    "newbalanceOrig": "float32",
    "oldbalanceDest": "float32",
    "newbalanceDest": "float32",
    "isFraud": "int8",
    "isFlaggedFraud": "int8",
}
usecols = ["step","type","amount","nameOrig","nameDest","isFraud","isFlaggedFraud"]

df = pd.read_csv(CSV_PATH, usecols=usecols, dtype=dtypes, nrows=N_ROWS)
df["step"] = df["step"].astype("int32")

# --- preprocessing ---
# 1) Keep only transaction types where fraud is possible
df = df[df["type"].isin(["CASH_OUT","TRANSFER"])].copy()

# 2) Remove self-transactions
df = df[df["nameOrig"] != df["nameDest"]].copy()

# 3) Hash IDs for privacy & smaller graph labels
df["nameOrig"] = df["nameOrig"].apply(hash_id)
df["nameDest"] = df["nameDest"].apply(hash_id)

# 4) Drop any negative/invalid amounts (defensive)
df = df[df["amount"] >= 0].copy()

print("✅ Loaded & preprocessed shape:", df.shape)
print("Txn types:", df["type"].unique())
print("Fraud rate (%):", round(100*df['isFraud'].mean(), 4))
display(df.head(5))


✅ Loaded & preprocessed shape: (2770409, 7)
Txn types: ['TRANSFER', 'CASH_OUT']
Categories (5, object): ['CASH_IN', 'CASH_OUT', 'DEBIT', 'PAYMENT', 'TRANSFER']
Fraud rate (%): 0.2965


,step,type,amount,nameOrig,nameDest,isFraud,isFlaggedFraud
2,1,TRANSFER,181.000000,93f0e0a7,4c28f12a,1,0
3,1,CASH_OUT,181.000000,34a01a4c,84dd08ef,1,0
15,1,CASH_OUT,229133.937500,63605bf2,95771a59,0,0
19,1,TRANSFER,215310.296875,c3a576e3,58b1a8c6,0,0
24,1,TRANSFER,311685.875000,89abb3f6,53474875,0,0


#4 - EDA — counts, types, amount stats

In [35]:
summary = {
    "rows": len(df),
    "steps_range": f"{int(df.step.min())} – {int(df.step.max())} hours",
    "unique_senders": df["nameOrig"].nunique(),
    "unique_receivers": df["nameDest"].nunique(),
    "fraud_rate_%": round(100*df["isFraud"].mean(), 4),
}
print("📊 Data Summary")
for k,v in summary.items(): print(f"  • {k}: {v}")

print("\n📦 Top transaction types:")
display(df["type"].value_counts().to_frame("count").head(10))

print("\n💰 Amount describe:")
display(df["amount"].describe(percentiles=[.5,.9,.99]).to_frame("amount_stats"))


📊 Data Summary
  • rows: 2770409
  • steps_range: 1 – 743 hours
  • unique_senders: 2767741
  • unique_receivers: 509523
  • fraud_rate_%: 0.2965

📦 Top transaction types:


,count
type,
CASH_OUT,2237500
TRANSFER,532909
CASH_IN,0
DEBIT,0
PAYMENT,0



💰 Amount describe:


,amount_stats
count,2.770409e+06
mean,3.175362e+05
std,8.858404e+05
min,0.000000e+00
50%,1.712609e+05
90%,5.448676e+05
99%,2.650036e+06
max,9.244552e+07


#5 - Filter a demo window (speed up graph build)

In [36]:
window_df = df[(df["step"] >= WINDOW_MIN_H) & (df["step"] <= WINDOW_MAX_H)].copy()
print(f"⏳ Window rows: {len(window_df)} within [{WINDOW_MIN_H}, {WINDOW_MAX_H}] hours")
display(window_df.head(5))


⏳ Window rows: 1045328 within [0, 200] hours


,step,type,amount,nameOrig,nameDest,isFraud,isFlaggedFraud
2,1,TRANSFER,181.000000,93f0e0a7,4c28f12a,1,0
3,1,CASH_OUT,181.000000,34a01a4c,84dd08ef,1,0
15,1,CASH_OUT,229133.937500,63605bf2,95771a59,0,0
19,1,TRANSFER,215310.296875,c3a576e3,58b1a8c6,0,0
24,1,TRANSFER,311685.875000,89abb3f6,53474875,0,0


#6 - Build graph (nodes: user, receiver; edges: txn with attributes)

In [37]:
G = nx.Graph()
USER, RECEIVER = "user", "receiver"

def upsert_txn_row(row):
    u = str(row["nameOrig"])
    r = str(row["nameDest"])
    if u not in G: G.add_node(u, ntype=USER)
    if r not in G: G.add_node(r, ntype=RECEIVER)
    G.add_edge(u, r, rel="txn", step=int(row["step"]), amount=float(row["amount"]), ttype=str(row["type"]))

window_df.apply(upsert_txn_row, axis=1)

print(f"🧠 Graph built: nodes={G.number_of_nodes():,} | edges={G.number_of_edges():,}")
print("\n🔎 Sample nodes:")
for n,data in list(G.nodes(data=True))[:5]: print(n, data)
print("\n🔎 Sample edges:")
for a,b,data in list(G.edges(data=True))[:5]: print(a, b, data)


🧠 Graph built: nodes=1,236,760 | edges=1,045,328

🔎 Sample nodes:
93f0e0a7 {'ntype': 'user'}
4c28f12a {'ntype': 'receiver'}
34a01a4c {'ntype': 'user'}
84dd08ef {'ntype': 'receiver'}
63605bf2 {'ntype': 'user'}

🔎 Sample edges:
93f0e0a7 4c28f12a {'rel': 'txn', 'step': 1, 'amount': 181.0, 'ttype': 'TRANSFER'}
4c28f12a 58b084f7 {'rel': 'txn', 'step': 6, 'amount': 109985.6484375, 'ttype': 'TRANSFER'}
4c28f12a e09c2277 {'rel': 'txn', 'step': 8, 'amount': 111622.390625, 'ttype': 'CASH_OUT'}
4c28f12a 59525801 {'rel': 'txn', 'step': 9, 'amount': 1447322.25, 'ttype': 'TRANSFER'}
4c28f12a a1cade47 {'rel': 'txn', 'step': 14, 'amount': 340825.5625, 'ttype': 'CASH_OUT'}


#7 - Helpers (ego nodes, safe metrics, motifs, louvain)

In [38]:
import math
from collections import defaultdict
import community as community_louvain  # python-louvain

def ego_nodes(G, seeds, k=2):
    nodes = set()
    for s in seeds:
        if s in G:
            nodes |= nx.ego_graph(G, s, radius=k).nodes
    return nodes

def safe_core_number(subG):
    if subG.number_of_edges() == 0: return 0
    try:
        core = nx.core_number(subG)
        return max(core.values())
    except Exception:
        return 0

def safe_pagerank(subG, node):
    if subG.number_of_edges() == 0 or node not in subG: return 0.0
    try:
        pr = nx.pagerank(subG, alpha=0.85, max_iter=100)
        return float(pr.get(node, 0.0))
    except Exception:
        return 0.0

def receiver_shared_degree(G, receiver_id):
    return float(G.degree(receiver_id)) if receiver_id in G else 0.0

def temporal_burst(edge_steps, window=24):
    if len(edge_steps) == 0: return 0.0
    latest = max(edge_steps)
    recent = sum(1 for s in edge_steps if latest - s <= window)
    return recent / max(1.0, len(edge_steps))

def louvain_features(subG):
    if subG.number_of_nodes() < 3 or subG.number_of_edges() == 0:
        return 1
    part = community_louvain.best_partition(subG)
    counts = defaultdict(int)
    for cid in part.values(): counts[cid] += 1
    return max(counts.values())  # community_size of dominant community

def uru_motif_count(subG):
    # Count receiver nodes with degree ≥ 2 (user–receiver–user motif)
    cnt = 0
    for n,d in subG.nodes(data=True):
        if d.get("ntype")==RECEIVER and subG.degree(n) >= 2:
            cnt += 1
    return float(cnt)

# Scoring weights (you may tune later; kept transparent for viva)
WEIGHTS_7D = {
    "component_size": 0.8,
    "shared_receiver_degree": 1.0,
    "kcore": 0.7,
    "pagerank_user": 0.6,
    "temporal_burst": 0.5,
    "community_size": 0.4,
    "motif_uru": 0.4,
}
WEIGHTS_30D = {
    "component_size": 0.7,
    "shared_receiver_degree": 0.9,
    "kcore": 0.7,
    "pagerank_user": 0.6,
    "temporal_burst": 0.3,
    "community_size": 0.4,
    "motif_uru": 0.4,
}

def _norm(name, val):
    if name in ("component_size","shared_receiver_degree","community_size","motif_uru"):
        return math.log1p(val)/5.0
    if name == "kcore":
        return min(val/10.0, 1.0)
    if name == "pagerank_user":
        return min(val*100.0, 1.0)
    if name == "temporal_burst":
        return min(val, 1.0)
    return 0.0

def score_features(feats: dict, weights: dict):
    contrib, z = {}, 0.0
    for k,w in weights.items():
        nv = _norm(k, feats.get(k,0.0))
        imp = w * nv
        contrib[k] = (feats.get(k,0.0), imp)
        z += imp
    score = 1/(1+math.exp(-z))
    reasons = sorted(
        [{"feature":k,"value":v,"impact":imp} for k,(v,imp) in contrib.items()],
        key=lambda x: abs(x["impact"]), reverse=True
    )[:5]
    return score, reasons


#8 - Extract features per txn in 7d & 30d windows (relative to txn step)

In [42]:
# Hot-fix: correct safe_pagerank call and re-run precision recompute

import time, math
import pandas as pd, numpy as np, networkx as nx
from tqdm import tqdm

# ---- make sure these exist (created earlier in the precision cell) ----
if "WEIGHTS_7D" not in globals():
    WEIGHTS_7D = {
        "component_size": 0.8, "shared_receiver_degree": 1.0, "kcore": 0.7,
        "pagerank_user": 0.6, "temporal_burst": 0.5, "community_size": 0.4, "motif_uru": 0.4,
    }
if "WEIGHTS_30D" not in globals():
    WEIGHTS_30D = {
        "component_size": 0.7, "shared_receiver_degree": 0.9, "kcore": 0.7,
        "pagerank_user": 0.6, "temporal_burst": 0.3, "community_size": 0.4, "motif_uru": 0.4,
    }

# ----------- correct helper definitions -----------
def safe_core_number(SG):
    if SG.number_of_edges() == 0:
        return 0.0
    try:
        core = nx.core_number(SG)
        return float(max(core.values()))
    except Exception:
        return 0.0

def safe_pagerank(SG, node):
    """Correct signature: (SG, node)"""
    if SG.number_of_edges() == 0 or node not in SG:
        return 0.0
    try:
        pr = nx.pagerank(SG, alpha=0.85, max_iter=100)
        return float(pr.get(node, 0.0))
    except Exception:
        return 0.0

def temporal_burst(edge_steps, window=24):
    if not edge_steps: return 0.0
    latest = max(edge_steps)
    recent = sum(1 for s in edge_steps if latest - s <= window)
    return float(recent) / float(max(1, len(edge_steps)))

def uru_motif_count(SG):
    cnt = 0
    for n,d in SG.nodes(data=True):
        if d.get("ntype") == "receiver" and SG.degree(n) >= 2:
            cnt += 1
    return float(cnt)

try:
    from collections import defaultdict
    import community as community_louvain
    def louvain_community_size(SG):
        if SG.number_of_nodes() < 3 or SG.number_of_edges() == 0:
            return 1.0
        part = community_louvain.best_partition(SG)
        counts = {}
        for cid in part.values():
            counts[cid] = counts.get(cid, 0) + 1
        return float(max(counts.values()))
except Exception:
    def louvain_community_size(SG):
        return 1.0

def extract_feats(SG, u, r):
    comp = len(max(nx.connected_components(SG), key=len)) if SG.number_of_nodes() else 1.0
    recv = float(SG.degree(r)) if r in SG else 0.0
    kc   = safe_core_number(SG)
    pr   = safe_pagerank(SG, u)     # <-- fixed call
    brst = temporal_burst([d.get("step",0) for _,_,d in SG.edges(data=True)], window=24)
    comm = louvain_community_size(SG)
    motifs = uru_motif_count(SG)
    return dict(
        component_size=comp, shared_receiver_degree=recv, kcore=kc,
        pagerank_user=pr, temporal_burst=brst, community_size=comm, motif_uru=motifs,
    )

def subgraph_for_window(G, seeds, step_min, step_max, k=2):
    SG = nx.Graph()
    add_node, add_edge = SG.add_node, SG.add_edge
    for a,b,d in G.edges(data=True):
        st = d.get("step",0)
        if step_min <= st <= step_max:
            if a not in SG: add_node(a, **G.nodes[a])
            if b not in SG: add_node(b, **G.nodes[b])
            add_edge(a,b, **d)
    # ego around the seeds
    nodes = set()
    for s in seeds:
        if s in SG:
            nodes |= nx.ego_graph(SG, s, radius=k).nodes
    return SG.subgraph(nodes).copy()

# ----------- re-run the precision loop safely -----------
# Use existing selection if present; else pick top-K by ring_score_7d or ring_score
if "topK" in globals() and len(topK):
    selection = topK.copy()
else:
    rank_col = "ring_score_7d" if "ring_score_7d" in feat.columns else "ring_score"
    K = min(500, len(feat))
    selection = feat.nlargest(K, rank_col).copy()

updates = []
SEVEN_D_HRS = 168
THIRTY_D_HRS = 720
K_HOP = 2

t0 = time.time()
for _, row in tqdm(selection.iterrows(), total=len(selection), desc="Precision recompute (hot-fix)"):
    u, r, s = str(row["txn_user"]), str(row["txn_receiver"]), int(row["step"])
    SG7  = subgraph_for_window(G, [u, r], s - SEVEN_D_HRS,  s, k=K_HOP)
    SG30 = subgraph_for_window(G, [u, r], s - THIRTY_D_HRS, s, k=K_HOP)
    f7, f30 = extract_feats(SG7, u, r), extract_feats(SG30, u, r)
    s7, rs7   = score_features(f7,  WEIGHTS_7D)
    s30, rs30 = score_features(f30, WEIGHTS_30D)
    updates.append({
        "idx": int(row["idx"]),
        "ring_score_7d": float(s7),
        "ring_score_30d": float(s30),
        "delta_score": float(s7 - s30),
        "f7_community_size": f7["community_size"],
        "f7_motif_uru": f7["motif_uru"],
        "reasons_7d": rs7,
        "reasons_30d": rs30,
        "precision_enhanced": True
    })

upd_df = pd.DataFrame(updates).set_index("idx")
feat = feat.set_index("idx")
for col in upd_df.columns:
    feat.loc[upd_df.index, col] = upd_df[col]
feat["ring_score"] = feat.get("ring_score_7d", feat.get("ring_score", np.nan))
feat["precision_enhanced"] = feat["precision_enhanced"].fillna(False)
feat = feat.reset_index()

print(f"✅ Precision step completed in {time.time()-t0:.1f}s (fixed)")
display(
    feat.sort_values("ring_score_7d" if "ring_score_7d" in feat.columns else "ring_score",
                     ascending=False).head(10)[
        ["idx","txn_user","txn_receiver","amount","type","isFraud",
         "ring_score_7d","ring_score_30d","delta_score","precision_enhanced"]
    ]
)


Precision recompute (hot-fix): 100%|██████████| 500/500 [46:17<00:00,  5.55s/it]

✅ Precision step completed in 2777.5s (fixed)



/tmp/ipython-input-3019283094.py:135: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  feat["precision_enhanced"] = feat["precision_enhanced"].fillna(False)


,idx,txn_user,txn_receiver,amount,type,isFraud,ring_score_7d,ring_score_30d,delta_score,precision_enhanced
15755,44276,fba8ec7a,a186c0a9,8.974354e+05,TRANSFER,0,0.947643,0.927174,0.020469,True
15120,43096,9a7ac41d,d07cdb7e,1.861411e+05,CASH_OUT,0,0.947643,0.927174,0.020469,True
12554,37057,60f5d32f,d07cdb7e,5.782684e+04,CASH_OUT,0,0.947643,0.927174,0.020469,True
12925,37902,01385738,d07cdb7e,2.467958e+05,CASH_OUT,0,0.947643,0.927174,0.020469,True
14091,41048,cead3b14,aaef0d74,1.995629e+06,TRANSFER,0,0.947643,0.927174,0.020469,True
12325,36615,e74f0726,aaef0d74,1.756959e+06,TRANSFER,0,0.947643,0.927174,0.020469,True
17145,47293,be638d8e,a186c0a9,3.090082e+05,CASH_OUT,0,0.947643,0.927174,0.020469,True
18743,50657,5b998372,d07cdb7e,2.741156e+05,CASH_OUT,0,0.947643,0.927174,0.020469,True
17045,47089,70c83038,aaef0d74,7.018435e+05,TRANSFER,0,0.947643,0.927174,0.020469,True
9472,30155,9640ab22,a186c0a9,3.887853e+05,CASH_OUT,0,0.946099,0.925268,0.020831,True


#9 - Add a simple confidence score & list top suspicious txns

In [43]:
# Confidence proxy: more edges in ego-subgraph and stable (|delta| small or large depending on logic)
# Here: confidence = min(1, 0.5 + 0.5*rank of (f7_component_size + f7_shared_receiver_degree))

feat = feat.copy()
feat["confidence"] = (feat["f7_component_size"] + feat["f7_shared_receiver_degree"])
feat["confidence"] = (feat["confidence"] - feat["confidence"].min()) / max(1e-9, (feat["confidence"].max()-feat["confidence"].min()))
feat["confidence"] = 0.5 + 0.5*feat["confidence"]

feat["ring_score"] = feat["ring_score_7d"]  # primary score to sort in demo
TOP_N = 20
top = feat.sort_values("ring_score", ascending=False).head(TOP_N)
print(f"🏅 Top {TOP_N} suspicious txns (by 7d score):")
display(top[["idx","txn_user","txn_receiver","amount","type","isFraud","ring_score","confidence"]])


🏅 Top 20 suspicious txns (by 7d score):


,idx,txn_user,txn_receiver,amount,type,isFraud,ring_score,confidence
15755,44276,fba8ec7a,a186c0a9,8.974354e+05,TRANSFER,0,0.947643,0.961538
15120,43096,9a7ac41d,d07cdb7e,1.861411e+05,CASH_OUT,0,0.947643,1.000000
12554,37057,60f5d32f,d07cdb7e,5.782684e+04,CASH_OUT,0,0.947643,1.000000
12925,37902,01385738,d07cdb7e,2.467958e+05,CASH_OUT,0,0.947643,1.000000
14091,41048,cead3b14,aaef0d74,1.995629e+06,TRANSFER,0,0.947643,0.969231
12325,36615,e74f0726,aaef0d74,1.756959e+06,TRANSFER,0,0.947643,0.969231
17145,47293,be638d8e,a186c0a9,3.090082e+05,CASH_OUT,0,0.947643,0.961538
18743,50657,5b998372,d07cdb7e,2.741156e+05,CASH_OUT,0,0.947643,1.000000
17045,47089,70c83038,aaef0d74,7.018435e+05,TRANSFER,0,0.947643,0.969231
9472,30155,9640ab22,a186c0a9,3.887853e+05,CASH_OUT,0,0.946099,0.961538


#10 - Evaluate ring_score with PR-AUC & Precision@K

In [44]:
from sklearn.metrics import average_precision_score

eval_df = feat.dropna(subset=["isFraud"]).copy()
y = eval_df["isFraud"].astype(int).values

def precision_at_k(scores, labels, k):
    idx = np.argsort(scores)[::-1][:k]
    return labels[idx].mean()

for col in ["ring_score_7d","ring_score_30d","delta_score"]:
    s = eval_df[col].values
    ap = average_precision_score(y, s)
    print(f"📈 {col}  PR-AUC: {ap:.4f}")
    for k in [50,100,200,500,1000]:
        if len(s) >= k:
            print(f"    • P@{k}: {precision_at_k(s,y,k):.3f}")


📈 ring_score_7d  PR-AUC: 0.0037
    • P@50: 0.000
    • P@100: 0.000
    • P@200: 0.000
    • P@500: 0.002
    • P@1000: 0.002
📈 ring_score_30d  PR-AUC: 0.0037
    • P@50: 0.000
    • P@100: 0.000
    • P@200: 0.000
    • P@500: 0.002
    • P@1000: 0.002
📈 delta_score  PR-AUC: 0.0052
    • P@50: 0.020
    • P@100: 0.010
    • P@200: 0.005
    • P@500: 0.002
    • P@1000: 0.001


#11 - Explain the #1 candidate with human-readable reasons

In [45]:
best = top.iloc[0]
ix = int(best["idx"])
reasons = best["reasons_7d"]
print(f"🔍 Top candidate idx={ix} | score={best['ring_score']:.3f} | isFraud={int(best['isFraud'])}")
print("🧾 Reasons:")
for r in reasons:
    # simple text template
    name, val, imp = r["feature"], r["value"], r["impact"]
    txt = {
        "component_size": "Large connected component around the sender",
        "shared_receiver_degree": "Receiver shared by many senders (possible mule hub)",
        "kcore": "High k-core index (dense subgraph)",
        "pagerank_user": "Sender has high centrality (influential in subgraph)",
        "temporal_burst": "Recent burst of transactions",
        "community_size": "Sender sits in a large community",
        "motif_uru": "Multiple users linked via same receivers (U–R–U motifs)",
    }.get(name, name)
    print(f"  • {txt}: value={val:.3f} (impact={imp:.3f})")


🔍 Top candidate idx=44276 | score=0.948 | isFraud=0
🧾 Reasons:
  • Receiver shared by many senders (possible mule hub): value=43.000 (impact=0.757)
  • Large connected component around the sender: value=44.000 (impact=0.609)
  • Sender has high centrality (influential in subgraph): value=0.013 (impact=0.600)
  • Recent burst of transactions: value=1.000 (impact=0.500)
  • Sender sits in a large community: value=44.000 (impact=0.305)


#12 - Export PyVis HTML for 7d & 30d views

In [47]:
# 12 - Export PyVis HTML for 7d & 30d views
from pyvis.network import Network
from IPython.display import display, HTML
import os

def export_pyvis_html(SG, out_name="fraud_ring.html", title="Subgraph"):
    net = Network(height="600px", width="100%", notebook=True)
    net.barnes_hut()  # prettier physics

    for n, data in SG.nodes(data=True):
        color = "#2b83ba" if data.get("ntype") == "user" else "#ffbf00"
        net.add_node(n, label=f"{data.get('ntype','')}:{n}", color=color)

    for a,b,data in SG.edges(data=True):
        net.add_edge(a, b, title=data.get("rel","txn"))

    net.show(out_name)
    return out_name

# pick top candidate
best = feat.sort_values("ring_score_7d", ascending=False).iloc[0]
seed_user = str(best["txn_user"])
seed_recv = str(best["txn_receiver"])
s = int(best["step"])

# build subgraphs
SG7  = subgraph_for_window(G, [seed_user, seed_recv], s-168, s)
SG30 = subgraph_for_window(G, [seed_user, seed_recv], s-720, s)

f7   = export_pyvis_html(SG7,  out_name="fraud_ring_7d.html",  title="7d Subgraph")
f30  = export_pyvis_html(SG30, out_name="fraud_ring_30d.html", title="30d Subgraph")

print("📁 Exported:", f7, "and", f30)
display(HTML(filename=f7))
display(HTML(filename=f30))


fraud_ring_7d.html
fraud_ring_30d.html
📁 Exported: fraud_ring_7d.html and fraud_ring_30d.html


#13 - Slice check — mean scores by type and amount decile (responsible AI evidence)

In [48]:
types = eval_df.groupby("type")["ring_score_7d"].mean().rename("mean_score_7d").to_frame()
types["fraud_rate"] = eval_df.groupby("type")["isFraud"].mean()
print("📎 By transaction type:")
display(types.sort_values("mean_score_7d", ascending=False))

# Amount deciles
eval_df = eval_df.copy()
eval_df["amount_decile"] = pd.qcut(eval_df["amount"].rank(method="first"), 10, labels=False)
dec = eval_df.groupby("amount_decile")[["ring_score_7d","isFraud"]].mean().rename(
    columns={"ring_score_7d":"mean_score_7d","isFraud":"fraud_rate"})
print("\n📎 By amount decile (0=lowest):")
display(dec)


📎 By transaction type:


,mean_score_7d,fraud_rate
type,,
TRANSFER,0.785830,0.009843
CASH_OUT,0.774887,0.003395



📎 By amount decile (0=lowest):


,mean_score_7d,fraud_rate
amount_decile,,
0,0.765160,0.0260
1,0.768283,0.0030
2,0.772376,0.0030
3,0.775442,0.0015
4,0.776411,0.0015
5,0.777707,0.0020
6,0.779586,0.0010
7,0.781603,0.0020
8,0.787282,0.0025


#14 - Ablation — rescore without certain features and compare P@K

In [51]:
# Title: Ablation — rescore without certain features and compare P@K (robust)

import numpy as np
from sklearn.metrics import average_precision_score

def rescore_without(features_df, drop_keys, weights=WEIGHTS_7D):
    """
    Recompute scores after removing some features (drop_keys).
    Falls back to 0.0 if an f7_* column is missing or NaN.
    """
    # keep only features that are NOT dropped
    cols = [k for k in weights.keys() if k not in drop_keys]
    w2   = {k: v for k, v in weights.items() if k in cols}

    f7_cols = [f"f7_{c}" for c in cols]
    # ensure all needed columns exist; if not, create them with zeros
    out = features_df.copy()
    for c in f7_cols:
        if c not in out.columns:
            out[c] = 0.0

    def row_score(r):
        # build features dict with safe 0.0 fallback and NaN -> 0.0
        feats = {c.replace("f7_", ""): float(r.get(c, 0.0)) for c in f7_cols}
        for k in feats:
            if feats[k] != feats[k]:  # NaN check
                feats[k] = 0.0
        s, _ = score_features(feats, w2)
        return s

    col_name = "score_no_" + "_".join(drop_keys)
    out[col_name] = out.apply(row_score, axis=1).fillna(0.0)
    return out, col_name

# ---- what to drop (one-at-a-time ablations) ----
drops = [
    ["kcore"],
    ["community_size"],
    ["motif_uru"],
    ["shared_receiver_degree"],
]

# y labels (ensure alignment with eval_df)
y = eval_df["isFraud"].astype(int).values

def precision_at_k(scores, labels, k):
    idx = np.argsort(scores)[::-1][:k]
    return labels[idx].mean()

print("🔬 Ablation vs base (ring_score_7d)")
for d in drops:
    tmp, col = rescore_without(eval_df, d, WEIGHTS_7D)
    # replace NaN with 0 for safety
    s = np.nan_to_num(tmp[col].values, nan=0.0)
    ap  = average_precision_score(y, s)
    p100 = precision_at_k(s, y, 100) if len(s) >= 100 else np.nan
    print(f"  • drop {d}: PR-AUC={ap:.4f} | P@100={p100:.3f}")


🔬 Ablation vs base (ring_score_7d)
  • drop ['kcore']: PR-AUC=0.0037 | P@100=0.000
  • drop ['community_size']: PR-AUC=0.0037 | P@100=0.000
  • drop ['motif_uru']: PR-AUC=0.0037 | P@100=0.000
  • drop ['shared_receiver_degree']: PR-AUC=0.0036 | P@100=0.000


#15 - Export CSV (scores) + save HTML files for UI

In [52]:
export_cols = ["idx","step","txn_user","txn_receiver","amount","type","isFraud",
               "ring_score_7d","ring_score_30d","delta_score","confidence"]
out_csv = "/content/ring_scores_window.csv"
feat[export_cols].to_csv(out_csv, index=False)
print("💾 Exported:", out_csv)

# Keep the HTMLs we generated (7d/30d)
print("📄 Subgraph HTML files saved in the Colab working dir: fraud_ring_7d.html / fraud_ring_30d.html")


💾 Exported: /content/ring_scores_window.csv
📄 Subgraph HTML files saved in the Colab working dir: fraud_ring_7d.html / fraud_ring_30d.html


#16 - Print a sample audit log row (for security & transparency slide)

In [53]:
import uuid, time, json
trace_id = str(uuid.uuid4())
audit = {
    "trace_id": trace_id,
    "when": int(time.time()),
    "endpoint": "/v1/graph/score",
    "request": {"txn_idx": int(best["idx"]), "window": "both", "k_hop": 2},
    "response": {
        "ring_score_7d": float(best["ring_score_7d"]),
        "ring_score_30d": float(best["ring_score_30d"]),
        "delta_score": float(best["delta_score"]),
        "confidence": float(best["confidence"]),
        "reasons_7d": best["reasons_7d"],
        "subgraph_ref": "fraud_ring_7d.html"
    },
    "caller": "demo-colab",
    "status": 200
}
print(json.dumps(audit, indent=2))


{
  "trace_id": "0e15c08e-5d7c-4fef-afaa-eece547170b7",
  "when": 1755972634,
  "endpoint": "/v1/graph/score",
  "request": {
    "txn_idx": 44276,
    "window": "both",
    "k_hop": 2
  },
  "response": {
    "ring_score_7d": 0.947642826020948,
    "ring_score_30d": 0.9271742323023938,
    "delta_score": 0.02046859371855414,
    "confidence": 0.9615384615384616,
    "reasons_7d": [
      {
        "feature": "shared_receiver_degree",
        "value": 43.0,
        "impact": 0.7568379267836522
      },
      {
        "feature": "component_size",
        "value": 44.0,
        "impact": 0.6090659983632511
      },
      {
        "feature": "pagerank_user",
        "value": 0.012528079538957083,
        "impact": 0.6
      },
      {
        "feature": "temporal_burst",
        "value": 1.0,
        "impact": 0.5
      },
      {
        "feature": "community_size",
        "value": 44.0,
        "impact": 0.3045329991816256
      }
    ],
    "subgraph_ref": "fraud_ring_7d.html"
  },


#17 - Responsible AI notes (printable)

In [54]:
notes = """
Responsible AI for Graph & Link Agent
-------------------------------------
• Privacy: PaySim is synthetic; in production hash/salt IDs. No PII stored.
• Fairness: Only structural/temporal behavior used (no demographic/geo). Slice checks by type/amount included.
• Explainability: Each score returns top-5 reasons + 7d/30d comparison + interactive subgraph viz.
• Transparency & Audit: Trace ID + request/response logging; exportable CSV for review.
• Human-in-the-loop: Scores are advisory; analysts review cases; their feedback can seed/propagate risk (configurable, deterministic).
"""
print(notes)



Responsible AI for Graph & Link Agent
-------------------------------------
• Privacy: PaySim is synthetic; in production hash/salt IDs. No PII stored.
• Fairness: Only structural/temporal behavior used (no demographic/geo). Slice checks by type/amount included.
• Explainability: Each score returns top-5 reasons + 7d/30d comparison + interactive subgraph viz.
• Transparency & Audit: Trace ID + request/response logging; exportable CSV for review.
• Human-in-the-loop: Scores are advisory; analysts review cases; their feedback can seed/propagate risk (configurable, deterministic).



In [55]:
from google.colab import files
files.download("fraud_ring_7d.html")
files.download("fraud_ring_30d.html")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [57]:
!cp /content/ring_scores_window.csv /content/drive/MyDrive/


In [58]:
!cp /content/sample_graph_score.json /content/drive/MyDrive/

cp: cannot stat '/content/sample_graph_score.json': No such file or directory
